<a href="https://colab.research.google.com/github/vignesh-potharaj/gen-ai/blob/main/Customer_Review_Analyzer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip uninstall -y torchaudio -q

In [ ]:
# Cell 1: Environment Setup and Dataset Loading for 7_2_Assessment (AI Customer Review Analyzer)

# 1. Install Hugging Face Transformers, Datasets, Accelerate, and Gemini SDK
!pip install -q -U transformers datasets accelerate google-genai torch

import os
import torch
import pandas as pd
from datasets import load_dataset
from google.colab import userdata
from google.genai import types, client
from google.genai.errors import ClientError

# 2. Setup Gemini API Client with Fallback logic for explanation steps
api_key = userdata.get('GEMINI_API_KEY')
ai = client.Client(api_key=api_key)

MODEL_CANDIDATES = [
    "gemini-3.6-flash",
    "gemini-3.5-flash",
    "gemini-3.1-flash-lite",
    "gemini-2.0-flash",
    "gemini-1.5-flash"
]

def generate_with_fallback(prompt, config=None):
    last_error = None
    for model_name in MODEL_CANDIDATES:
        try:
            res = ai.models.generate_content(
                model=model_name,
                contents=prompt,
                config=config
            )
            print(f"[Success] Response generated using active model: {model_name}\n")
            return res
        except ClientError as e:
            if e.code == 404 or "NOT_FOUND" in str(e):
                last_error = e
                continue
            raise e
    raise RuntimeError(f"All model candidates failed. Last error: {last_error}")

# 3. Load IMDB Dataset using complete Hugging Face repository ID ('stanfordnlp/imdb')
print("Loading IMDB Dataset for Customer Review Fine-Tuning...")
imdb_dataset = load_dataset("stanfordnlp/imdb")

# Select a small subset for fast execution inside Colab free tier GPU
train_data = imdb_dataset["train"].shuffle(seed=42).select(range(800))
test_data = imdb_dataset["test"].shuffle(seed=42).select(range(200))

print("\n=== ASSESSMENT 7_2: DATASET INGESTION COMPLETE ===")
print(f"Training Sample Size: {len(train_data)} reviews")
print(f"Testing Sample Size: {len(test_data)} reviews")
print(f"Sample Review Text: {train_data[0]['text'][:150]}...")
print(f"Sample Label (0 = Negative, 1 = Positive): {train_data[0]['label']}")

Loading IMDB Dataset for Customer Review Fine-Tuning...

=== ASSESSMENT 7_2: DATASET INGESTION COMPLETE ===
Training Sample Size: 800 reviews
Testing Sample Size: 200 reviews
Sample Review Text: There is no relation at all between Fortier and Profiler but the fact that both are police series about violent crimes. Profiler looks crispy, Fortier...
Sample Label (0 = Negative, 1 = Positive): 1


In [ ]:
# Cell 2: Tokenization and Model Initialization for 7_2_Assessment (AI Customer Review Analyzer)

from transformers import AutoTokenizer, AutoModelForSequenceClassification

# 1. Pretrained model checkpoint (DistilBERT)
MODEL_CHECKPOINT = "distilbert-base-uncased"

print(f"Loading tokenizer for: {MODEL_CHECKPOINT}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

# 2. Tokenization function
def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=256)

# 3. Map tokenization across datasets
print("Tokenizing customer review datasets...")
tokenized_train = train_data.map(preprocess_function, batched=True)
tokenized_test = test_data.map(preprocess_function, batched=True)

# 4. Load Pretrained Sequence Classification Model
print(f"Loading pretrained sequence classifier for: {MODEL_CHECKPOINT}...")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=2
)

print("\n=== ASSESSMENT 7_2: DISTILBERT TOKENIZATION & INITIALIZATION COMPLETE ===")
print(f"Tokenizer Vocab Size: {tokenizer.vocab_size}")
print(f"Model Architecture: {model.config.architectures[0]}")

Loading tokenizer for: distilbert-base-uncased...
Tokenizing customer review datasets...
Loading pretrained sequence classifier for: distilbert-base-uncased...


ModuleNotFoundError: Could not import module 'DistilBertForSequenceClassification'. Are this object's requirements defined correctly?

In [ ]:
# Cell 3: Fine-Tuning Execution & Model Inference for 7_2_Assessment

import torch
import numpy as np
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding
from google.genai import types

# 1. Dynamic padding data collator
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# 2. Configure training parameters for fast Colab execution
training_args = TrainingArguments(
    output_dir="./review_analyzer_results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,
    eval_strategy="no",
    save_strategy="no",
    logging_steps=10,
    fp16=torch.cuda.is_available(),
)

# 3. Initialize Hugging Face Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

# 4. Execute Fine-Tuning Process
print("Starting DistilBERT fine-tuning on customer reviews...\n")
trainer.train()
print("\n=== FINE-TUNING PROCESS COMPLETE ===")

# 5. Sentiment Inference Function
def predict_review_sentiment(review_text):
    inputs = tokenizer(review_text, return_tensors="pt", truncation=True, max_length=256)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    inputs = {k: v.to(device) for k, v in inputs.items()}
    model.to(device)

    model.eval()
    with torch.no_grad():
        logits = model(**inputs).logits

    predicted_class = torch.argmax(logits, dim=-1).item()
    return "Positive" if predicted_class == 1 else "Negative"

# 6. Test Fine-Tuned Model on Unseen Customer Reviews
test_reviews = [
    "The customer service was outstanding and the item arrived earlier than expected!",
    "Extremely disappointed. The packaging was broken and the product stopped working after one day."
]

print("\n=== FINE-TUNED MODEL INFERENCE RESULTS ===")
for review in test_reviews:
    sentiment = predict_review_sentiment(review)
    print(f"Review: \"{review}\"\n  └─ Predicted Sentiment: {sentiment}\n")